#  Audio Branch (UC1) Training & Evaluation

Trains the audio emotion classifier (three feature streams — hand-crafted acoustic features, Wav2Vec2 embeddings, and a Whisper+RoBERTa semantic stream with emotion-word scrubbing — combined via a learned attention gate) on a fixed 30/6/6 subject-independent split (seed = 42), and exports the saved test-set predictions used by `Fusion.ipynb`.

**Phases:** 0 (install) → 1 (unzip + metadata) → 2 (feature extraction) → 3 (data loading + split) → 4 (model) → 5 (training) → 6 (evaluation) → 7 (export for fusion). Each phase is resumable — re-running from the top skips work that's already been done (checkpointed to Drive).

##**PHASE 0 · INSTALL DEPENDENCIES**

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 0 · INSTALL DEPENDENCIES                              ║
# ║  Run once per Colab session. Restart runtime if prompted.    ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

PACKAGES = [
    "openai-whisper",
    "transformers>=4.35.0",
    "accelerate>=0.21.0",
    "librosa>=0.10.0",
    "h5py>=3.9.0",
    "scikit-learn>=1.3.0",
    "statsmodels>=0.14.0",
    "seaborn>=0.13.0",
    "tqdm>=4.66.0",
    "joblib>=1.3.0",
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages installed successfully.")

# Quick version check
import torch, transformers, librosa, sklearn
print(f"\n📦 Key Package Versions:")
print(f"   PyTorch      : {torch.__version__}")
print(f"   Transformers : {transformers.__version__}")
print(f"   Librosa      : {librosa.__version__}")
print(f"   Scikit-learn : {sklearn.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

✅ All packages installed successfully.

📦 Key Package Versions:
   PyTorch      : 2.11.0+cpu
   Transformers : 5.10.2
   Librosa      : 0.11.0
   Scikit-learn : 1.6.1
   CUDA Available: False


##**MOUNT GOOGLE DRIVE**

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  MOUNT GOOGLE DRIVE                                          ║
# ╚══════════════════════════════════════════════════════════════╝

from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted at /content/drive")

Mounted at /content/drive
✅ Google Drive mounted at /content/drive


## **GLOBAL CONFIGURATION — EDIT ONLY THIS CELL**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  GLOBAL CONFIGURATION — EDIT ONLY THIS CELL                  ║
# ╚══════════════════════════════════════════════════════════════╝

from pathlib import Path

class Config:
    # ─── Google Drive Paths ───────────────────────────────────
    DRIVE_ROOT          = Path("/content/drive/MyDrive/THESIS")
    AUDIO_ROOT          = DRIVE_ROOT / "Audio Results"          

    DRIVE_AUDIO_ZIP     = DRIVE_ROOT / "Audio.zip"          
    DRIVE_FEATURES_DIR  = AUDIO_ROOT / "Features"
    DRIVE_MODELS_DIR    = AUDIO_ROOT / "Models/audio"
    DRIVE_FIGURES_DIR   = AUDIO_ROOT / "Figures/audio"
    DRIVE_RESULTS_DIR   = AUDIO_ROOT / "Results/audio"

    # ─── Local Colab Paths (fast SSD) ─────────────────────────
    LOCAL_AUDIO_DIR     = Path("/content/audio_data/Audio")
    LOCAL_FEATURES_DIR  = Path("/content/features_audio")
    LOCAL_METADATA      = Path("/content/audio_metadata.csv")
    LOCAL_CACHE_DIR     = Path("/content/features_audio/cache")
    LOCAL_CHECKPOINT_DB = Path("/content/features_audio/checkpoint.db")

    # ─── Audio Processing ─────────────────────────────────────
    SAMPLE_RATE  = 16000     # Hz — EAV dataset standard
    DURATION     = 20        # seconds per trial
    TRIM_TOP_DB  = 25        # silence trim threshold
    MAX_SAMPLES  = SAMPLE_RATE * DURATION

    # ─── Pre-trained Models ───────────────────────────────────
    WAV2VEC_MODEL   = "facebook/wav2vec2-large-960h"   # 1024-dim
    WHISPER_MODEL   = "base"
    ROBERTA_MODEL   = "cardiffnlp/twitter-roberta-base-sentiment"  # 768-dim

    # ─── Traditional Feature Settings ─────────────────────────
    N_MFCC      = 40
    N_CHROMA    = 12
    N_MELS      = 128
    N_CONTRAST  = 7
    N_TONNETZ   = 6
    HOP_LENGTH  = 512
    N_FFT       = 2048

    # ─── Emotion Setup ────────────────────────────────────────
    LABEL_MAP    = {"H": 0, "S": 1, "A": 2, "C": 3, "N": 4}
    REVERSE_MAP  = {0: "H", 1: "S", 2: "A", 3: "C", 4: "N"}
    EMOTION_NAMES = {
        "H": "Happiness", "S": "Sadness",
        "A": "Angry",     "C": "Calmness", "N": "Neutral"
    }
    EMOTION_FULL = ["Happiness", "Sadness", "Angry", "Calmness", "Neutral"]
    EMOTION_MAP = {
        "happiness": "H", "happy": "H",
        "sadness": "S",   "sad": "S",
        "angry": "A",     "anger": "A",
        "calmness": "C",  "calm": "C",
        "neutral": "N"
    }

    # ─── Subject Split ────────────────────────────────────────
    N_SUBJECTS      = 42
    TRAIN_SUBJECTS  = 30   # 71.4%
    VAL_SUBJECTS    = 6    # 14.3%
    TEST_SUBJECTS   = 6    # 14.3%
    SEED            = 42   # Fixed for reproducibility

    # ─── Training Hyperparameters ─────────────────────────────
    BATCH_SIZE          = 64
    EPOCHS              = 100
    LEARNING_RATE       = 3e-4
    WEIGHT_DECAY        = 1e-4
    EARLY_STOP_PATIENCE = 15
    LR_PATIENCE         = 5
    LR_FACTOR           = 0.5
    GRAD_CLIP_NORM      = 1.0

    # ─── Architecture ─────────────────────────────────────────
    HIDDEN_DIM   = 512
    DROPOUT      = 0.40
    NUM_CLASSES  = 5

    # ─── Publication ──────────────────────────────────────────
    FIGURE_DPI   = 300
    FIGURE_FMT   = "png"   # or "pdf" for LaTeX

    # ─── Debug ────────────────────────────────────────────────
    TEST_MODE    = False   # True → only process 5 files (quick test)
    TEST_N       = 5

    @classmethod
    def setup_all_dirs(cls):
        for d in [cls.DRIVE_FEATURES_DIR, cls.DRIVE_MODELS_DIR,
                  cls.DRIVE_FIGURES_DIR,  cls.DRIVE_RESULTS_DIR,
                  cls.LOCAL_FEATURES_DIR, cls.LOCAL_CACHE_DIR]:
            d.mkdir(parents=True, exist_ok=True)
        print("✅ All directories created/verified.")

Config.setup_all_dirs()
print("\n⚙️  Configuration loaded. Key paths:")
print(f"   Audio ZIP     : {Config.DRIVE_AUDIO_ZIP}")
print(f"   Features H5   : {Config.DRIVE_FEATURES_DIR / 'audio_features.h5'}")
print(f"   Model weights : {Config.DRIVE_MODELS_DIR / 'best_audio_model.pt'}")
print(f"   Figures       : {Config.DRIVE_FIGURES_DIR}")
print(f"   Seed          : {Config.SEED}")

✅ All directories created/verified.

⚙️  Configuration loaded. Key paths:
   Audio ZIP     : /content/drive/MyDrive/THESIS/Audio.zip
   Features H5   : /content/drive/MyDrive/THESIS/Audio Results/Features/audio_features.h5
   Model weights : /content/drive/MyDrive/THESIS/Audio Results/Models/audio/best_audio_model.pt
   Figures       : /content/drive/MyDrive/THESIS/Audio Results/Figures/audio
   Seed          : 42


## **REPRODUCIBILITY SETUP + LOGGING**

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  REPRODUCIBILITY SETUP + LOGGING                             ║
# ╚══════════════════════════════════════════════════════════════╝

import random, os, sys, logging, time
import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(Config.SEED)

def get_logger(name: str = "SENTIRA-Audio") -> logging.Logger:
    log_path = Config.LOCAL_FEATURES_DIR / "logs"
    log_path.mkdir(exist_ok=True)
    log_file = log_path / f"run_{time.strftime('%Y%m%d_%H%M%S')}.log"

    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s",
                            datefmt="%H:%M:%S")
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt)
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger

LOG = get_logger()
LOG.info(f"Device: {DEVICE}")
LOG.info(f"Seed  : {Config.SEED}")
if torch.cuda.is_available():
    LOG.info(f"GPU   : {torch.cuda.get_device_name(0)}")
    LOG.info(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

10:21:08 | INFO | Device: cpu


INFO:SENTIRA-Audio:Device: cpu


10:21:08 | INFO | Seed  : 42


INFO:SENTIRA-Audio:Seed  : 42


## **PHASE 1A · UNZIP DATASET**

In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 1A · UNZIP DATASET                                    ║
# ╚══════════════════════════════════════════════════════════════╝

import os, shutil
from pathlib import Path
from tqdm import tqdm
import pandas as pd

# Check if already unzipped
if Config.LOCAL_AUDIO_DIR.exists():
    subject_dirs = [d for d in Config.LOCAL_AUDIO_DIR.iterdir() if d.is_dir()]
    if len(subject_dirs) >= 40:
        LOG.info(f"✅ Audio already unzipped → {len(subject_dirs)} subjects found. Skipping unzip.")
    else:
        LOG.warning(f"⚠️  Only {len(subject_dirs)} subjects found. Re-unzipping...")
        shutil.rmtree(Config.LOCAL_AUDIO_DIR.parent, ignore_errors=True)
        os.makedirs(Config.LOCAL_AUDIO_DIR.parent, exist_ok=True)
        os.system(f'unzip -q "{Config.DRIVE_AUDIO_ZIP}" -d "/content/audio_data"')
        LOG.info("✅ Unzip complete.")
else:
    LOG.info(f"📦 Unzipping from {Config.DRIVE_AUDIO_ZIP} ...")
    os.makedirs("/content/audio_data", exist_ok=True)
    ret = os.system(f'unzip -q "{Config.DRIVE_AUDIO_ZIP}" -d "/content/audio_data"')
    if ret != 0:
        LOG.error("❌ Unzip FAILED. Check that Audio.zip exists on Drive at THESIS/Audio.zip")
    else:
        LOG.info("✅ Unzip complete.")

# ── Verify Structure ──
subject_dirs = sorted([d for d in Config.LOCAL_AUDIO_DIR.iterdir()
                        if d.is_dir() and d.name.lower().startswith("subject")],
                       key=lambda d: int("".join(filter(str.isdigit, d.name)) or "0"))
LOG.info(f"📂 Found {len(subject_dirs)} subject folders")

# Sample verification
if subject_dirs:
    sample = subject_dirs[0]
    audio_sub = sample / "Audio"
    search_path = audio_sub if audio_sub.exists() else sample
    wav_files = list(search_path.glob("*.wav")) + list(search_path.glob("*.WAV"))
    LOG.info(f"📄 Sample subject: {sample.name} → {len(wav_files)} audio files")
    if wav_files:
        LOG.info(f"   Example: {wav_files[0].name}")

10:21:08 | INFO | 📦 Unzipping from /content/drive/MyDrive/THESIS/Audio.zip ...


INFO:SENTIRA-Audio:📦 Unzipping from /content/drive/MyDrive/THESIS/Audio.zip ...


10:21:29 | ERROR | ❌ Unzip FAILED. Check that Audio.zip exists on Drive at THESIS/Audio.zip


ERROR:SENTIRA-Audio:❌ Unzip FAILED. Check that Audio.zip exists on Drive at THESIS/Audio.zip


10:21:29 | INFO | 📂 Found 2 subject folders


INFO:SENTIRA-Audio:📂 Found 2 subject folders


10:21:29 | INFO | 📄 Sample subject: Subject1 → 100 audio files


INFO:SENTIRA-Audio:📄 Sample subject: Subject1 → 100 audio files


10:21:29 | INFO |    Example: 048_Trial_04_Speaking_Sadness_aud.wav


INFO:SENTIRA-Audio:   Example: 048_Trial_04_Speaking_Sadness_aud.wav


##**PHASE 1B · BUILD METADATA CSV**

In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 1B · BUILD METADATA CSV                               ║
# ╚══════════════════════════════════════════════════════════════╝

import re

EMOTION_PATTERN = re.compile(
    r'\b(happiness|happy|sadness|sad|angry|anger|calmness|calm|neutral)\b',
    flags=re.IGNORECASE
)

def detect_emotion_from_filename(filename: str) -> str:
    """Extract emotion code from trial filename."""
    fname = Path(filename).stem.lower()
    for word, code in Config.EMOTION_MAP.items():
        if word in fname:
            return code
    # Fallback: check for single-letter codes at word boundaries
    for code in ["H", "S", "A", "C", "N"]:
        if re.search(rf'\b{code}\b', fname.upper()):
            return code
    return "N"  # default neutral if undetectable

def build_metadata() -> pd.DataFrame:
    """Scan all subject folders and build metadata CSV."""
    if Config.LOCAL_METADATA.exists():
        df = pd.read_csv(Config.LOCAL_METADATA)
        LOG.info(f"✅ Metadata already exists → {len(df)} samples. Skipping rebuild.")
        return df

    LOG.info("🔍 Building metadata CSV...")
    records = []
    subject_dirs = sorted(
        [d for d in Config.LOCAL_AUDIO_DIR.iterdir()
         if d.is_dir() and d.name.lower().startswith("subject")],
        key=lambda d: int("".join(filter(str.isdigit, d.name)) or "0")
    )

    for sub_dir in tqdm(subject_dirs, desc="Scanning subjects"):
        subject_id = sub_dir.name  # e.g., "Subject1"
        subject_num = "".join(filter(str.isdigit, subject_id))

        # Support both Subject1/file.wav and Subject1/Audio/file.wav
        audio_dir = sub_dir / "Audio"
        if not audio_dir.exists():
            audio_dir = sub_dir

        wav_files = sorted(
            list(audio_dir.glob("*.wav")) + list(audio_dir.glob("*.WAV"))
        )

        for fp in wav_files:
            emotion_code = detect_emotion_from_filename(fp.name)
            trial_num = "".join(filter(str.isdigit, fp.stem.split("_")[-1] or "0"))
            records.append({
                "subject_id":  subject_id,
                "subject_num": int(subject_num) if subject_num else 0,
                "trial_id":    fp.stem,
                "emotion_code": emotion_code,
                "emotion_name": Config.EMOTION_NAMES.get(emotion_code, "Unknown"),
                "filepath":    str(fp),
            })

    df = pd.DataFrame(records)
    df.to_csv(Config.LOCAL_METADATA, index=False)
    LOG.info(f"✅ Metadata saved → {len(df)} total samples")
    return df

df_meta = build_metadata()

# ── Distribution Check ──
print("\n📊 Dataset Distribution:")
print(df_meta.groupby("emotion_name").size().rename("Count").to_string())
print(f"\n   Total samples : {len(df_meta)}")
print(f"   Total subjects: {df_meta['subject_id'].nunique()}")
print(f"   Avg/subject   : {len(df_meta)/df_meta['subject_id'].nunique():.0f}")

10:21:29 | INFO | 🔍 Building metadata CSV...


INFO:SENTIRA-Audio:🔍 Building metadata CSV...
Scanning subjects: 100%|██████████| 2/2 [00:00<00:00, 171.07it/s]

10:21:29 | INFO | ✅ Metadata saved → 150 total samples



INFO:SENTIRA-Audio:✅ Metadata saved → 150 total samples



📊 Dataset Distribution:
emotion_name
Angry        32
Calmness     32
Happiness    28
Neutral      30
Sadness      28

   Total samples : 150
   Total subjects: 2
   Avg/subject   : 75


## **PHASE 2 GATE — SKIP IF HDF5 EXISTS**

In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 2 GATE — SKIP IF HDF5 EXISTS                          ║
# ╚══════════════════════════════════════════════════════════════╝

import shutil

H5_NAME       = "audio_features.h5"
H5_DRIVE_PATH = Config.DRIVE_FEATURES_DIR / H5_NAME
H5_LOCAL_PATH = Config.LOCAL_FEATURES_DIR / H5_NAME

SKIP_EXTRACTION = False

if H5_DRIVE_PATH.exists():
    LOG.info(f"✅ HDF5 found on Drive at: {H5_DRIVE_PATH}")
    LOG.info(f"   File size: {H5_DRIVE_PATH.stat().st_size / 1e9:.2f} GB")
    if not H5_LOCAL_PATH.exists():
        LOG.info("📥 Copying HDF5 from Drive to local for fast access...")
        shutil.copy(H5_DRIVE_PATH, H5_LOCAL_PATH)
        LOG.info("✅ Copy complete.")
    else:
        LOG.info("✅ HDF5 already local. Ready to use.")
    SKIP_EXTRACTION = True

elif H5_LOCAL_PATH.exists():
    LOG.info(f"✅ HDF5 found locally at: {H5_LOCAL_PATH}")
    SKIP_EXTRACTION = True

else:
    LOG.info("⚙️  No HDF5 file found. Feature extraction pipeline will run.")
    LOG.info(f"   Processing {len(df_meta)} files from {df_meta['subject_id'].nunique()} subjects.")
    SKIP_EXTRACTION = False

if Config.TEST_MODE:
    LOG.warning(f"⚠️  TEST MODE ACTIVE — Only first {Config.TEST_N} files will be processed.")

10:21:29 | INFO | ✅ HDF5 found on Drive at: /content/drive/MyDrive/THESIS/Audio Results/Features/audio_features.h5


INFO:SENTIRA-Audio:✅ HDF5 found on Drive at: /content/drive/MyDrive/THESIS/Audio Results/Features/audio_features.h5


10:21:29 | INFO |    File size: 0.08 GB


INFO:SENTIRA-Audio:   File size: 0.08 GB


10:21:29 | INFO | 📥 Copying HDF5 from Drive to local for fast access...


INFO:SENTIRA-Audio:📥 Copying HDF5 from Drive to local for fast access...


10:21:31 | INFO | ✅ Copy complete.


INFO:SENTIRA-Audio:✅ Copy complete.


## **CHECKPOINT MANAGER (SQLite — GPU/CPU Resume)**  

In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CHECKPOINT MANAGER (SQLite — GPU/CPU Resume)                ║
# ╚══════════════════════════════════════════════════════════════╝

import sqlite3
from pathlib import Path
from typing import Optional

class CheckpointManager:
    """
    SQLite-backed per-file checkpoint manager.
    Enables resumable extraction across Colab disconnects.
    The DB is saved on Drive after each batch for persistence.
    """
    def __init__(self, local_db: Path, drive_db: Path):
        self.local_db = local_db
        self.drive_db = drive_db
        self._sync_from_drive()
        self._init_db()

    def _sync_from_drive(self):
        """Copy checkpoint DB from Drive if exists (resume after disconnect)."""
        if self.drive_db.exists() and not self.local_db.exists():
            shutil.copy(self.drive_db, self.local_db)
            LOG.info(f"📥 Resumed checkpoint DB from Drive ({self.drive_db.stat().st_size/1024:.0f} KB)")

    def _init_db(self):
        with sqlite3.connect(self.local_db) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS processed_files (
                    filepath   TEXT PRIMARY KEY,
                    cache_path TEXT,
                    status     TEXT,
                    timestamp  TEXT DEFAULT CURRENT_TIMESTAMP
                )
            """)
            conn.commit()

    def is_done(self, filepath: str) -> bool:
        with sqlite3.connect(self.local_db) as conn:
            row = conn.execute(
                "SELECT status FROM processed_files WHERE filepath=?", (filepath,)
            ).fetchone()
            return row is not None and row[0] == "SUCCESS"

    def mark_success(self, filepath: str, cache_path: str):
        with sqlite3.connect(self.local_db) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO processed_files VALUES (?,?,?,CURRENT_TIMESTAMP)",
                (filepath, cache_path, "SUCCESS")
            )
            conn.commit()

    def mark_failed(self, filepath: str, error: str):
        with sqlite3.connect(self.local_db) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO processed_files VALUES (?,?,?,CURRENT_TIMESTAMP)",
                (filepath, "", f"FAILED:{error[:200]}")
            )
            conn.commit()

    def sync_to_drive(self):
        """Backup checkpoint DB to Drive."""
        shutil.copy(self.local_db, self.drive_db)

    def stats(self) -> dict:
        with sqlite3.connect(self.local_db) as conn:
            total   = conn.execute("SELECT COUNT(*) FROM processed_files").fetchone()[0]
            success = conn.execute("SELECT COUNT(*) FROM processed_files WHERE status='SUCCESS'").fetchone()[0]
            failed  = conn.execute("SELECT COUNT(*) FROM processed_files WHERE status LIKE 'FAILED%'").fetchone()[0]
        return {"total": total, "success": success, "failed": failed}

## **PHASE 2 · MULTI-STREAM FEATURE EXTRACTOR**

In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 2 · MULTI-STREAM FEATURE EXTRACTOR                    ║
# ╚══════════════════════════════════════════════════════════════╝

import warnings, re
import numpy as np
import librosa, librosa.feature, librosa.effects
import torch
from transformers import (
    Wav2Vec2Processor, Wav2Vec2Model,
    AutoTokenizer, AutoModelForSequenceClassification
)
import whisper
from typing import Tuple

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SCRUB_PATTERN = re.compile(
    r'\b(happy|sad|angry|neutral|calm|calmness|sadness|happiness|anger)\b',
    re.IGNORECASE
)

def load_audio(filepath: str) -> Optional[np.ndarray]:
    """Load and preprocess audio file to 16kHz mono, exactly 20 seconds."""
    try:
        y, sr = librosa.load(filepath, sr=Config.SAMPLE_RATE,
                             duration=Config.DURATION, mono=True)
        # Trim silence
        y_trim, _ = librosa.effects.trim(y, top_db=Config.TRIM_TOP_DB)
        if len(y_trim) < Config.SAMPLE_RATE * 0.5:
            y_trim = y  # Revert if over-trimmed
        # Pad or truncate to exactly MAX_SAMPLES
        if len(y_trim) < Config.MAX_SAMPLES:
            y_trim = np.pad(y_trim, (0, Config.MAX_SAMPLES - len(y_trim)))
        else:
            y_trim = y_trim[:Config.MAX_SAMPLES]
        # Normalize to [-1, 1]
        max_val = np.abs(y_trim).max()
        if max_val > 0:
            y_trim = y_trim / max_val
        return y_trim.astype(np.float32)
    except Exception as e:
        LOG.error(f"Audio load failed: {filepath} → {e}")
        return None

def extract_traditional(audio: np.ndarray) -> np.ndarray:
    """
    Extract comprehensive acoustic feature vector.
    Returns ~558-dim float32 vector.
    """
    sr = Config.SAMPLE_RATE
    feats = []

    # MFCC (40 coefficients) + Delta + Delta-Delta
    mfcc   = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=Config.N_MFCC,
                                   hop_length=Config.HOP_LENGTH, n_fft=Config.N_FFT)
    d_mfcc  = librosa.feature.delta(mfcc)
    dd_mfcc = librosa.feature.delta(mfcc, order=2)

    for m in [mfcc, d_mfcc, dd_mfcc]:
        feats.extend([m.mean(axis=1), m.std(axis=1)])  # 40+40 each = 240 total

    # Chroma (12 bins)
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr, hop_length=Config.HOP_LENGTH)
    feats.extend([chroma.mean(axis=1), chroma.std(axis=1)])  # 24

    # Mel Spectrogram (128 bins → dB scale)
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=Config.N_MELS,
                                          hop_length=Config.HOP_LENGTH, n_fft=Config.N_FFT)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feats.extend([mel_db.mean(axis=1), mel_db.std(axis=1)])  # 256

    # Spectral Features (scalar mean+std each)
    for fn in [
        librosa.feature.spectral_centroid,
        librosa.feature.spectral_bandwidth,
        librosa.feature.spectral_rolloff,
        librosa.feature.spectral_flatness,
    ]:
        try:
            feat = fn(y=audio, sr=sr)
            feats.extend([[feat.mean()], [feat.std()]])      # 2 each = 8 total
        except Exception:
            feats.extend([[0.0], [0.0]])

    # Spectral Contrast (7 bands)
    try:
        contrast = librosa.feature.spectral_contrast(y=audio, sr=sr,
                                                      hop_length=Config.HOP_LENGTH)
        feats.extend([contrast.mean(axis=1), contrast.std(axis=1)])  # 14
    except Exception:
        feats.extend([np.zeros(7), np.zeros(7)])

    # Tonnetz (6 components — pitch class relationships)
    try:
        tonnetz = librosa.feature.tonnetz(
            y=librosa.effects.harmonic(audio), sr=sr
        )
        feats.extend([tonnetz.mean(axis=1), tonnetz.std(axis=1)])  # 12
    except Exception:
        feats.extend([np.zeros(6), np.zeros(6)])

    # ZCR + RMS Energy
    zcr = librosa.feature.zero_crossing_rate(audio)
    rms = librosa.feature.rms(y=audio)
    feats.extend([[zcr.mean()], [zcr.std()], [rms.mean()], [rms.std()]])  # 4

    return np.concatenate([np.ravel(f) for f in feats]).astype(np.float32)


class MultiStreamExtractor:
    """
    Loads and manages the three pre-trained models.
    Extracts Wav2Vec2 + Semantic (Whisper+RoBERTa) features.
    """
    def __init__(self):
        self.device = DEVICE
        self._models_loaded = False

    def load_models(self):
        LOG.info("📥 Loading Wav2Vec2-Large ...")
        self.w2v_processor = Wav2Vec2Processor.from_pretrained(Config.WAV2VEC_MODEL)
        self.w2v_model     = Wav2Vec2Model.from_pretrained(Config.WAV2VEC_MODEL).to(self.device).eval()
        LOG.info("📥 Loading Whisper ...")
        self.whisper_model = whisper.load_model(Config.WHISPER_MODEL, device=self.device)
        LOG.info("📥 Loading RoBERTa ...")
        self.roberta_tokenizer = AutoTokenizer.from_pretrained(Config.ROBERTA_MODEL)
        self.roberta_model     = AutoModelForSequenceClassification.from_pretrained(
            Config.ROBERTA_MODEL
        ).to(self.device).eval()
        self._models_loaded = True
        LOG.info("✅ All models loaded.")

    def extract_wav2vec(self, audio: np.ndarray) -> np.ndarray:
        """Mean-pooled Wav2Vec2-Large last hidden state → 1024-dim."""
        inputs = self.w2v_processor(
            audio, sampling_rate=Config.SAMPLE_RATE,
            return_tensors="pt", padding=True
        )
        with torch.no_grad():
            out = self.w2v_model(inputs.input_values.to(self.device))
            emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return emb.astype(np.float32)

    def extract_semantic(self, audio: np.ndarray) -> Tuple[np.ndarray, str]:
        """
        Whisper ASR → scrub emotion keywords → RoBERTa last-layer [CLS] → 768-dim.
        Scrubbing prevents emotion word leakage into semantic embeddings.
        """
        try:
            result = self.whisper_model.transcribe(
                audio.astype(np.float32),
                fp16=torch.cuda.is_available()
            )
            text = result.get("text", "").strip()
        except Exception:
            text = ""

        if not text or len(text) < 3:
            return np.zeros(768, dtype=np.float32), "[NO_TRANSCRIPT]"

        # Scrub emotion keyword to prevent data leakage
        scrubbed = SCRUB_PATTERN.sub("[EMO]", text)

        try:
            inputs = self.roberta_tokenizer(
                scrubbed, return_tensors="pt",
                truncation=True, max_length=128
            ).to(self.device)
            with torch.no_grad():
                out = self.roberta_model(**inputs, output_hidden_states=True)
                # Last hidden state, [CLS] token
                emb = out.hidden_states[-1][:, 0, :].squeeze().cpu().numpy()
            return emb.astype(np.float32), scrubbed
        except Exception as e:
            LOG.warning(f"RoBERTa failed: {e}")
            return np.zeros(768, dtype=np.float32), scrubbed

LOG.info("✅ Feature extractor classes defined.")

10:21:59 | INFO | ✅ Feature extractor classes defined.


INFO:SENTIRA-Audio:✅ Feature extractor classes defined.


## **PHASE 2 · RUN EXTRACTION PIPELINE**

In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 2 · RUN EXTRACTION PIPELINE                           ║
# ╚══════════════════════════════════════════════════════════════╝

import time
from tqdm import tqdm
from typing import Optional
import pandas as pd

if not SKIP_EXTRACTION:
    # ── Setup ──
    df = df_meta.copy()
    if Config.TEST_MODE:
        df = df.head(Config.TEST_N)
        LOG.warning(f"TEST MODE: Processing only {Config.TEST_N} files.")

    DRIVE_DB = Config.DRIVE_MODELS_DIR / "checkpoint.db"
    ckpt = CheckpointManager(Config.LOCAL_CHECKPOINT_DB, DRIVE_DB)
    extractor = MultiStreamExtractor()
    extractor.load_models()

    # ── Extraction Loop ──
    LOG.info(f"🚀 Starting extraction: {len(df)} files")
    t0 = time.time()
    success_count = 0
    fail_count    = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting Features"):
        fp = str(row["filepath"])

        # Build cache key and path
        cache_key  = f"{row['subject_id']}_{row['trial_id']}"
        cache_path = Config.LOCAL_CACHE_DIR / f"{cache_key}.npz"

        # Skip if already processed
        if ckpt.is_done(fp) and cache_path.exists():
            success_count += 1
            continue

        # Load audio
        audio = load_audio(fp)
        if audio is None:
            ckpt.mark_failed(fp, "AudioLoadFailed")
            fail_count += 1
            continue

        try:
            # Extract all streams
            trad = extract_traditional(audio)
            w2v  = extractor.extract_wav2vec(audio)
            sem, transcript = extractor.extract_semantic(audio)

            # Save compressed .npz cache
            np.savez_compressed(
                cache_path,
                traditional  = trad,
                wav2vec      = w2v,
                semantic     = sem,
                transcript   = np.array([transcript]),
            )
            ckpt.mark_success(fp, str(cache_path))
            success_count += 1

        except Exception as e:
            LOG.error(f"Extraction failed for {fp}: {e}")
            ckpt.mark_failed(fp, str(e))
            fail_count += 1

        # Periodic GPU memory cleanup + Drive backup
        if (idx + 1) % 100 == 0:
            torch.cuda.empty_cache()
            ckpt.sync_to_drive()
            elapsed = time.time() - t0
            rate    = success_count / elapsed
            eta     = (len(df) - success_count - fail_count) / rate if rate > 0 else 0
            LOG.info(f"Progress: {success_count}/{len(df)} | "
                     f"Failed: {fail_count} | ETA: {eta/60:.1f} min")

    # Final sync
    ckpt.sync_to_drive()
    elapsed_total = (time.time() - t0) / 60
    stats = ckpt.stats()
    LOG.info(f"✅ Extraction complete in {elapsed_total:.1f} min")
    LOG.info(f"   Success: {stats['success']} | Failed: {stats['failed']}")

else:
    LOG.info("⏭️  SKIPPING EXTRACTION (HDF5 already exists)")

10:21:59 | INFO | ⏭️  SKIPPING EXTRACTION (HDF5 already exists)


INFO:SENTIRA-Audio:⏭️  SKIPPING EXTRACTION (HDF5 already exists)


## **PHASE 2 · COMPILE HDF5 + SAVE TO DRIVE**

In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 2 · COMPILE HDF5 + SAVE TO DRIVE                      ║
# ╚══════════════════════════════════════════════════════════════╝

import h5py

if not SKIP_EXTRACTION:
    LOG.info("📦 Compiling all .npz cache files into HDF5...")
    df = df_meta.copy()
    if Config.TEST_MODE:
        df = df.head(Config.TEST_N)

    total_added = 0
    total_skipped = 0

    with h5py.File(H5_LOCAL_PATH, "w") as hf:
        hf.attrs["description"]  = "SENTIRA Audio Emotion Recognition Features"
        hf.attrs["dataset"]      = "EAV"
        hf.attrs["n_subjects"]   = df["subject_id"].nunique()
        hf.attrs["n_samples"]    = len(df)
        hf.attrs["emotions"]     = ",".join(Config.LABEL_MAP.keys())
        hf.attrs["seed"]         = Config.SEED
        hf.attrs["created"]      = time.strftime("%Y-%m-%d %H:%M:%S")
        hf.attrs["wav2vec_model"] = Config.WAV2VEC_MODEL
        hf.attrs["whisper_model"] = Config.WHISPER_MODEL
        hf.attrs["roberta_model"] = Config.ROBERTA_MODEL

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Compiling HDF5"):
            cache_key  = f"{row['subject_id']}_{row['trial_id']}"
            cache_path = Config.LOCAL_CACHE_DIR / f"{cache_key}.npz"

            if not cache_path.exists():
                total_skipped += 1
                continue

            data = np.load(cache_path, allow_pickle=True)
            grp = hf.create_group(cache_key)
            grp.create_dataset("traditional", data=data["traditional"], compression="gzip", compression_opts=6)
            grp.create_dataset("wav2vec",     data=data["wav2vec"],     compression="gzip", compression_opts=6)
            grp.create_dataset("semantic",    data=data["semantic"],    compression="gzip", compression_opts=6)
            grp.attrs["emotion"]     = str(row["emotion_code"])
            grp.attrs["emotion_name"]= str(row["emotion_name"])
            grp.attrs["subject"]     = str(row["subject_id"])
            grp.attrs["subject_num"] = int(row["subject_num"])
            grp.attrs["trial_id"]    = str(row["trial_id"])
            grp.attrs["filepath"]    = str(row["filepath"])
            transcript = str(data["transcript"][0]) if "transcript" in data else ""
            grp.attrs["transcript"]  = transcript
            total_added += 1

    LOG.info(f"✅ HDF5 compiled: {total_added} samples, {total_skipped} skipped")
    LOG.info(f"   File size: {H5_LOCAL_PATH.stat().st_size / 1e9:.3f} GB")

    # ── Copy to Google Drive ──
    LOG.info(f"☁️  Copying HDF5 to Drive: {H5_DRIVE_PATH}")
    shutil.copy(H5_LOCAL_PATH, H5_DRIVE_PATH)
    LOG.info(f"✅ Saved to Drive. Size: {H5_DRIVE_PATH.stat().st_size / 1e6:.1f} MB")

# ── Quick HDF5 Verification ──
LOG.info("\n🔍 HDF5 Verification:")
with h5py.File(H5_LOCAL_PATH, "r") as hf:
    keys = list(hf.keys())
    first = hf[keys[0]]
    LOG.info(f"   Total entries : {len(keys)}")
    LOG.info(f"   Traditional   : {first['traditional'].shape}")
    LOG.info(f"   Wav2Vec2      : {first['wav2vec'].shape}")
    LOG.info(f"   Semantic      : {first['semantic'].shape}")
    emotion_counts = {}
    for k in keys:
        emo = hf[k].attrs.get("emotion", "?")
        emotion_counts[emo] = emotion_counts.get(emo, 0) + 1
    LOG.info(f"   Emotion dist  : {emotion_counts}")

10:21:59 | INFO | 
🔍 HDF5 Verification:


INFO:SENTIRA-Audio:
🔍 HDF5 Verification:


10:21:59 | INFO |    Total entries : 4200


INFO:SENTIRA-Audio:   Total entries : 4200


10:21:59 | INFO |    Traditional   : (558,)


...

INFO:SENTIRA-Audio:   Wav2Vec2      : (1024,)


10:21:59 | INFO |    Semantic      : (768,)


INFO:SENTIRA-Audio:   Semantic      : (768,)


10:22:03 | INFO |    Emotion dist  : {'N': 840, 'A': 840, 'C': 840, 'S': 840, 'H': 840}


INFO:SENTIRA-Audio:   Emotion dist  : {'N': 840, 'A': 840, 'C': 840, 'S': 840, 'H': 840}


## **PHASE 3 · LOAD DATA + SUBJECT-INDEPENDENT SPLIT**

In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 3 · LOAD DATA + SUBJECT-INDEPENDENT SPLIT             ║
# ╚══════════════════════════════════════════════════════════════╝

import h5py, json, random
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import pickle

SPLIT_DRIVE_PATH  = Config.DRIVE_MODELS_DIR / "subject_split.json"
SCALER_DRIVE_PATH = Config.DRIVE_MODELS_DIR / "scalers.pkl"
SCALER_LOCAL_PATH = Config.LOCAL_FEATURES_DIR / "scalers.pkl"

def load_hdf5_data(h5_path: Path) -> dict:
    """Load all features from HDF5 into a subject-keyed dict."""
    LOG.info(f"📥 Loading features from: {h5_path}")
    data = {}  # { subject_id: {trad, w2v, sem, labels} }

    with h5py.File(h5_path, "r") as hf:
        for key in hf.keys():
            grp     = hf[key]
            subject = grp.attrs.get("subject", "Unknown")
            emo     = grp.attrs.get("emotion", "N")
            label   = Config.LABEL_MAP.get(emo, 4)

            if subject not in data:
                data[subject] = {"trad": [], "w2v": [], "sem": [], "labels": [],
                                  "keys": [], "emotions": []}
            data[subject]["trad"].append(grp["traditional"][:])
            data[subject]["w2v"].append(grp["wav2vec"][:])
            data[subject]["sem"].append(grp["semantic"][:])
            data[subject]["labels"].append(label)
            data[subject]["keys"].append(key)
            data[subject]["emotions"].append(emo)

    LOG.info(f"✅ Loaded {len(data)} subjects")
    return data

def make_or_load_split(subjects: list) -> dict:
    """
    Reproducible subject split. Saved to Drive so it never changes
    between sessions — critical for valid train/test separation.
    """
    if SPLIT_DRIVE_PATH.exists():
        with open(SPLIT_DRIVE_PATH) as f:
            split = json.load(f)
        LOG.info(f"✅ Loaded existing split from Drive (reproducibility lock).")
        return split

    # New split
    rng = random.Random(Config.SEED)
    shuffled = subjects.copy()
    rng.shuffle(shuffled)
    split = {
        "train": shuffled[:Config.TRAIN_SUBJECTS],
        "val":   shuffled[Config.TRAIN_SUBJECTS:Config.TRAIN_SUBJECTS + Config.VAL_SUBJECTS],
        "test":  shuffled[Config.TRAIN_SUBJECTS + Config.VAL_SUBJECTS:],
        "seed":  Config.SEED,
        "created": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    with open(SPLIT_DRIVE_PATH, "w") as f:
        json.dump(split, f, indent=2)
    LOG.info(f"✅ New split created and saved to Drive.")
    return split

def compile_split(data: dict, subjects: list):
    """Concatenate features for a list of subjects."""
    trad, w2v, sem, labels = [], [], [], []
    for sub in subjects:
        if sub not in data:
            LOG.warning(f"Subject {sub} not found in HDF5. Skipping.")
            continue
        trad.extend(data[sub]["trad"])
        w2v.extend(data[sub]["w2v"])
        sem.extend(data[sub]["sem"])
        labels.extend(data[sub]["labels"])
    return (np.array(trad, dtype=np.float32),
            np.array(w2v,  dtype=np.float32),
            np.array(sem,  dtype=np.float32),
            np.array(labels, dtype=np.int64))

# ── Execute ──
subject_data = load_hdf5_data(H5_LOCAL_PATH)
all_subjects = list(subject_data.keys())
split_info   = make_or_load_split(all_subjects)

X_train = compile_split(subject_data, split_info["train"])
X_val   = compile_split(subject_data, split_info["val"])
X_test  = compile_split(subject_data, split_info["test"])

LOG.info(f"\n📊 Split Summary:")
LOG.info(f"   Train : {len(X_train[3])} samples ({len(split_info['train'])} subjects)")
LOG.info(f"   Val   : {len(X_val[3])} samples ({len(split_info['val'])} subjects)")
LOG.info(f"   Test  : {len(X_test[3])} samples ({len(split_info['test'])} subjects)")

# ── Feature Dimensions (Auto-detected) ──
TRAD_DIM = X_train[0].shape[1]
W2V_DIM  = X_train[1].shape[1]
SEM_DIM  = X_train[2].shape[1]
LOG.info(f"\n📐 Feature Dimensions (auto-detected):")
LOG.info(f"   Traditional : {TRAD_DIM}")
LOG.info(f"   Wav2Vec2    : {W2V_DIM}")
LOG.info(f"   Semantic    : {SEM_DIM}")
LOG.info(f"   Total       : {TRAD_DIM + W2V_DIM + SEM_DIM}")

10:22:03 | INFO | 📥 Loading features from: /content/features_audio/audio_features.h5


INFO:SENTIRA-Audio:📥 Loading features from: /content/features_audio/audio_features.h5


10:22:14 | INFO | ✅ Loaded 42 subjects


INFO:SENTIRA-Audio:✅ Loaded 42 subjects


10:22:15 | INFO | ✅ Loaded existing split from Drive (reproducibility lock).


...

INFO:SENTIRA-Audio:   Wav2Vec2    : 1024


10:22:15 | INFO |    Semantic    : 768


INFO:SENTIRA-Audio:   Semantic    : 768


10:22:15 | INFO |    Total       : 2350


INFO:SENTIRA-Audio:   Total       : 2350


## **PHASE 3 · NORMALIZE FEATURES (No Train/Test Leakage)**

In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 3 · NORMALIZE FEATURES (No Train/Test Leakage)        ║
# ╚══════════════════════════════════════════════════════════════╝

if SCALER_DRIVE_PATH.exists():
    # ── Load existing scalers (same session or resumed) ──
    LOG.info("✅ Loading scalers from Drive (resuming session)...")
    with open(SCALER_DRIVE_PATH, "rb") as f:
        scalers = pickle.load(f)
    tr_trad = scalers["trad"].transform(X_train[0])
    tr_w2v  = scalers["w2v"].transform(X_train[1])
    tr_sem  = scalers["sem"].transform(X_train[2])
else:
    # ── Fit scalers on TRAINING data only ──
    LOG.info("⚖️  Fitting StandardScalers on training data only...")
    scalers = {
        "trad": StandardScaler(),
        "w2v":  StandardScaler(),
        "sem":  StandardScaler(),
    }
    tr_trad = scalers["trad"].fit_transform(X_train[0])
    tr_w2v  = scalers["w2v"].fit_transform(X_train[1])
    tr_sem  = scalers["sem"].fit_transform(X_train[2])
    with open(SCALER_DRIVE_PATH, "wb") as f:
        pickle.dump(scalers, f)
    shutil.copy(SCALER_DRIVE_PATH, SCALER_LOCAL_PATH)
    LOG.info(f"✅ Scalers saved to Drive: {SCALER_DRIVE_PATH}")

# ── Apply to Val/Test ──
va_trad = scalers["trad"].transform(X_val[0])
va_w2v  = scalers["w2v"].transform(X_val[1])
va_sem  = scalers["sem"].transform(X_val[2])

te_trad = scalers["trad"].transform(X_test[0])
te_w2v  = scalers["w2v"].transform(X_test[1])
te_sem  = scalers["sem"].transform(X_test[2])

train_labels = X_train[3]
val_labels   = X_val[3]
test_labels  = X_test[3]

LOG.info("✅ Normalization complete (fitted on train, applied to val/test).")
LOG.info(f"   Train features mean: trad={tr_trad.mean():.4f}, w2v={tr_w2v.mean():.4f}")

10:22:15 | INFO | ✅ Loading scalers from Drive (resuming session)...


INFO:SENTIRA-Audio:✅ Loading scalers from Drive (resuming session)...


10:22:16 | INFO | ✅ Normalization complete (fitted on train, applied to val/test).


INFO:SENTIRA-Audio:✅ Normalization complete (fitted on train, applied to val/test).


10:22:16 | INFO |    Train features mean: trad=0.0000, w2v=-0.0000


INFO:SENTIRA-Audio:   Train features mean: trad=0.0000, w2v=-0.0000


## **PHASE 3 · PYTORCH DATASET + DATALOADERS**

In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 3 · PYTORCH DATASET + DATALOADERS                     ║
# ╚══════════════════════════════════════════════════════════════╝

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class EmotionDataset(Dataset):
    """PyTorch Dataset wrapping three feature streams + labels."""
    def __init__(self, trad, w2v, sem, labels):
        self.trad   = torch.from_numpy(trad).float()
        self.w2v    = torch.from_numpy(w2v).float()
        self.sem    = torch.from_numpy(sem).float()
        self.labels = torch.from_numpy(labels).long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.trad[idx], self.w2v[idx], self.sem[idx], self.labels[idx]

train_dataset = EmotionDataset(tr_trad, tr_w2v, tr_sem, train_labels)
val_dataset   = EmotionDataset(va_trad, va_w2v, va_sem, val_labels)
test_dataset  = EmotionDataset(te_trad, te_w2v, te_sem, test_labels)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

LOG.info(f"✅ DataLoaders ready:")
LOG.info(f"   Train: {len(train_loader)} batches × {Config.BATCH_SIZE}")
LOG.info(f"   Val  : {len(val_loader)} batches")
LOG.info(f"   Test : {len(test_loader)} batches")

10:22:16 | INFO | ✅ DataLoaders ready:


INFO:SENTIRA-Audio:✅ DataLoaders ready:


10:22:16 | INFO |    Train: 47 batches × 64


INFO:SENTIRA-Audio:   Train: 47 batches × 64


10:22:16 | INFO |    Val  : 10 batches


INFO:SENTIRA-Audio:   Val  : 10 batches


10:22:16 | INFO |    Test : 10 batches


INFO:SENTIRA-Audio:   Test : 10 batches


## **PHASE 4 · ATTENTION FUSION MODEL**

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 4 · ATTENTION FUSION MODEL                            ║
# ╚══════════════════════════════════════════════════════════════╝

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

class BranchEncoder(nn.Module):
    """
    Single-branch encoder: Linear → BN → GELU → Dropout → Linear → BN → GELU.
    Transforms arbitrary-dim features into shared hidden_dim space.
    """
    def __init__(self, in_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.75),
        )

    def forward(self, x):
        return self.net(x)


class AttentionFusionModel(nn.Module):
    """
    SENTIRA Audio Branch — Multi-stream Attention Fusion Classifier.

    Three branches (Traditional, Wav2Vec2, Semantic) are encoded into
    a shared space, then fused using learned cross-branch attention weights.

    Features for thesis:
    - Auto-detects input dimensions from data
    - Modality dropout during training for robustness
    - Exposes softmax probabilities for SENTIRA FusionEngine (UC4-UC7)
    - get_confidence() for confidence-weighted decision-level fusion
    """
    def __init__(
        self,
        trad_dim:   int = 558,
        w2v_dim:    int = 1024,
        sem_dim:    int = 768,
        hidden_dim: int = 512,
        num_classes:int = 5,
        dropout:    float = 0.40,
    ):
        super().__init__()
        self.trad_dim   = trad_dim
        self.w2v_dim    = w2v_dim
        self.sem_dim    = sem_dim
        self.hidden_dim = hidden_dim

        # Branch encoders
        self.trad_enc = BranchEncoder(trad_dim, hidden_dim, dropout)
        self.w2v_enc  = BranchEncoder(w2v_dim,  hidden_dim, dropout)
        self.sem_enc  = BranchEncoder(sem_dim,  hidden_dim, dropout)

        # Cross-branch attention gate: learns which stream to trust
        self.attn_gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 3),
            nn.Softmax(dim=1)          # 3 normalized weights
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout * 0.75),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.50),
            nn.Linear(128, num_classes),
        )

        # Weight initialization
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(
        self,
        trad: torch.Tensor,
        w2v:  torch.Tensor,
        sem:  torch.Tensor,
        return_attn: bool = False,
    ) -> torch.Tensor:
        # Modality dropout during training (robustness → fusion preparation)
        if self.training:
            if torch.rand(1).item() < 0.15:
                trad = torch.zeros_like(trad)  # Drop traditional
            if torch.rand(1).item() < 0.10:
                sem  = torch.zeros_like(sem)   # Drop semantic

        # Encode each branch
        h_t = self.trad_enc(trad)  # [B, H]
        h_w = self.w2v_enc(w2v)    # [B, H]
        h_s = self.sem_enc(sem)    # [B, H]

        # Compute attention weights
        cat     = torch.cat([h_t, h_w, h_s], dim=1)  # [B, 3H]
        weights = self.attn_gate(cat)                  # [B, 3]

        # Weighted fusion
        fused = (
            h_t * weights[:, 0:1] +
            h_w * weights[:, 1:2] +
            h_s * weights[:, 2:3]
        )                                              # [B, H]

        logits = self.classifier(fused)               # [B, num_classes]

        if return_attn:
            return logits, weights
        return logits

    # ─── Fusion-Ready Inference ───────────────────────────────
    @torch.no_grad()
    def get_softmax_probs(self, trad, w2v, sem) -> torch.Tensor:
        """Returns softmax probability vector [B, 5] — for FusionEngine."""
        self.eval()
        logits = self.forward(trad, w2v, sem)
        return torch.softmax(logits, dim=1)

    @torch.no_grad()
    def get_confidence(self, trad, w2v, sem) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (confidence, predicted_class) — for confidence-weighted fusion."""
        probs = self.get_softmax_probs(trad, w2v, sem)
        conf, cls = probs.max(dim=1)
        return conf, cls

    @torch.no_grad()
    def get_attention_weights(self, trad, w2v, sem) -> torch.Tensor:
        """Returns attention weights per branch [B, 3] — for visualization."""
        self.eval()
        _, weights = self.forward(trad, w2v, sem, return_attn=True)
        return weights


# ── Instantiate & Inspect ──
model = AttentionFusionModel(
    trad_dim    = TRAD_DIM,
    w2v_dim     = W2V_DIM,
    sem_dim     = SEM_DIM,
    hidden_dim  = Config.HIDDEN_DIM,
    num_classes = Config.NUM_CLASSES,
    dropout     = Config.DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
LOG.info(f"✅ AttentionFusionModel instantiated.")
LOG.info(f"   Trainable parameters: {n_params:,} ({n_params/1e6:.2f}M)")
LOG.info(f"   Input dims: trad={TRAD_DIM}, w2v={W2V_DIM}, sem={SEM_DIM}")
LOG.info(f"   Hidden dim: {Config.HIDDEN_DIM}")
print(model)

10:22:16 | INFO | ✅ AttentionFusionModel instantiated.


INFO:SENTIRA-Audio:✅ AttentionFusionModel instantiated.


10:22:16 | INFO |    Trainable parameters: 4,552,968 (4.55M)


INFO:SENTIRA-Audio:   Trainable parameters: 4,552,968 (4.55M)


10:22:16 | INFO |    Input dims: trad=558, w2v=1024, sem=768


INFO:SENTIRA-Audio:   Input dims: trad=558, w2v=1024, sem=768


10:22:16 | INFO |    Hidden dim: 512


INFO:SENTIRA-Audio:   Hidden dim: 512


AttentionFusionModel(
  (trad_enc): BranchEncoder(
    (net): Sequential(
      (0): Linear(in_features=558, out_features=1024, bias=True)
      (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.4, inplace=False)
      (4): Linear(in_features=1024, out_features=512, bias=True)
      (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): GELU(approximate='none')
      (7): Dropout(p=0.30000000000000004, inplace=False)
    )
  )
  (w2v_enc): BranchEncoder(
    (net): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.4, inplace=False)
      (4): Linear(in_features=1024, out_features=512, bias=True)
      (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running

## **PHASE 5 · TRAINING ORCHESTRATOR (RESUMABLE)**

In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 5 · TRAINING ORCHESTRATOR (RESUMABLE)                 ║
# ╚══════════════════════════════════════════════════════════════╝

import torch.optim as optim
import json, time
from pathlib import Path

MODEL_DRIVE_PATH   = Config.DRIVE_MODELS_DIR / "best_audio_model.pt"
MODEL_LOCAL_PATH   = Config.LOCAL_FEATURES_DIR / "best_audio_model.pt"
HISTORY_DRIVE_PATH = Config.DRIVE_MODELS_DIR / "training_history.json"

# ── Check for existing checkpoint ──
SKIP_TRAINING = False
if MODEL_DRIVE_PATH.exists():
    LOG.info(f"✅ Best model found on Drive: {MODEL_DRIVE_PATH}")
    LOG.info("   Loading model and SKIPPING training → proceeding to Evaluation.")
    shutil.copy(MODEL_DRIVE_PATH, MODEL_LOCAL_PATH)
    model.load_state_dict(torch.load(MODEL_LOCAL_PATH, map_location=DEVICE))
    model.eval()
    SKIP_TRAINING = True
    if HISTORY_DRIVE_PATH.exists():
        with open(HISTORY_DRIVE_PATH) as f:
            training_history = json.load(f)
        LOG.info(f"   Loaded training history ({len(training_history['train_acc'])} epochs)")
    else:
        training_history = None

if not SKIP_TRAINING:
    LOG.info("🚀 Starting training from scratch...")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=Config.LEARNING_RATE,
        weight_decay=Config.WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max",
        factor=Config.LR_FACTOR,
        patience=Config.LR_PATIENCE
    )

    best_val_acc  = 0.0
    patience_ctr  = 0
    training_history = {
        "train_loss": [], "train_acc": [],
        "val_loss":   [], "val_acc":   [],
        "lr_history": []
    }

    for epoch in range(1, Config.EPOCHS + 1):
        # ── Train ──
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for trad, w2v, sem, labels in train_loader:
            trad   = trad.to(DEVICE, non_blocking=True)
            w2v    = w2v.to(DEVICE,  non_blocking=True)
            sem    = sem.to(DEVICE,  non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(trad, w2v, sem)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP_NORM)
            optimizer.step()

            train_loss    += loss.item() * labels.size(0)
            preds          = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

        avg_train_loss = train_loss / train_total
        avg_train_acc  = 100.0 * train_correct / train_total

        # ── Validate ──
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for trad, w2v, sem, labels in val_loader:
                trad   = trad.to(DEVICE, non_blocking=True)
                w2v    = w2v.to(DEVICE,  non_blocking=True)
                sem    = sem.to(DEVICE,  non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                logits = model(trad, w2v, sem)
                loss   = criterion(logits, labels)
                val_loss    += loss.item() * labels.size(0)
                preds        = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)

        avg_val_loss = val_loss / val_total
        avg_val_acc  = 100.0 * val_correct / val_total
        current_lr   = optimizer.param_groups[0]["lr"]

        # ── Log ──
        training_history["train_loss"].append(avg_train_loss)
        training_history["train_acc"].append(avg_train_acc)
        training_history["val_loss"].append(avg_val_loss)
        training_history["val_acc"].append(avg_val_acc)
        training_history["lr_history"].append(current_lr)

        LOG.info(
            f"Epoch {epoch:3d}/{Config.EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f}  Acc: {avg_train_acc:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f}  Acc: {avg_val_acc:.2f}% | "
            f"LR: {current_lr:.2e}"
        )

        scheduler.step(avg_val_acc)

        # ── Checkpoint ──
        if avg_val_acc > best_val_acc:
            best_val_acc  = avg_val_acc
            patience_ctr  = 0
            torch.save(model.state_dict(), MODEL_LOCAL_PATH)
            shutil.copy(MODEL_LOCAL_PATH, MODEL_DRIVE_PATH)
            LOG.info(f"   💾 New best → {best_val_acc:.2f}% (saved to Drive)")
        else:
            patience_ctr += 1

        # ── Save history every epoch (resume-safe) ──
        with open(HISTORY_DRIVE_PATH, "w") as f:
            json.dump(training_history, f)

        # ── Early stopping ──
        if patience_ctr >= Config.EARLY_STOP_PATIENCE:
            LOG.info(f"🛑 Early stopping after {epoch} epochs (patience={Config.EARLY_STOP_PATIENCE})")
            break

    # Load best model for evaluation
    model.load_state_dict(torch.load(MODEL_DRIVE_PATH, map_location=DEVICE))
    model.eval()
    LOG.info(f"\n✅ Training complete. Best Val Accuracy: {best_val_acc:.2f}%")

10:22:16 | INFO | ✅ Best model found on Drive: /content/drive/MyDrive/THESIS/Audio Results/Models/audio/best_audio_model.pt


INFO:SENTIRA-Audio:✅ Best model found on Drive: /content/drive/MyDrive/THESIS/Audio Results/Models/audio/best_audio_model.pt


10:22:16 | INFO |    Loading model and SKIPPING training → proceeding to Evaluation.


INFO:SENTIRA-Audio:   Loading model and SKIPPING training → proceeding to Evaluation.


10:22:18 | INFO |    Loaded training history (38 epochs)


INFO:SENTIRA-Audio:   Loaded training history (38 epochs)


## **PHASE 5 · TRAINING CURVES (300 DPI)**

In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 5 · TRAINING CURVES (300 DPI)                         ║
# ╚══════════════════════════════════════════════════════════════╝

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

Config.DRIVE_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if training_history and len(training_history["train_acc"]) > 1:
    epochs_ran = list(range(1, len(training_history["train_acc"]) + 1))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("SENTIRA UC1 — Audio Emotion Recognition | Training Curves",
                 fontsize=14, fontweight="bold", y=1.01)

    # Accuracy
    ax = axes[0]
    ax.plot(epochs_ran, training_history["train_acc"], "b-o", ms=2, label="Train")
    ax.plot(epochs_ran, training_history["val_acc"],   "r-o", ms=2, label="Validation")
    ax.set_xlabel("Epoch");  ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy");  ax.legend();  ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))

    # Loss
    ax = axes[1]
    ax.plot(epochs_ran, training_history["train_loss"], "b-o", ms=2, label="Train")
    ax.plot(epochs_ran, training_history["val_loss"],   "r-o", ms=2, label="Validation")
    ax.set_xlabel("Epoch");  ax.set_ylabel("Loss (CE + Label Smoothing)")
    ax.set_title("Loss");  ax.legend();  ax.grid(True, alpha=0.3)

    # Learning Rate
    ax = axes[2]
    ax.semilogy(epochs_ran, training_history["lr_history"], "g-o", ms=2)
    ax.set_xlabel("Epoch");  ax.set_ylabel("Learning Rate")
    ax.set_title("Learning Rate Schedule");  ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out_path = Config.DRIVE_FIGURES_DIR / "training_curves.png"
    plt.savefig(out_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
    plt.show()
    LOG.info(f"✅ Training curves saved: {out_path}")
else:
    LOG.info("ℹ️  Training was skipped (model loaded from Drive). Plotting history if available.")

10:22:26 | INFO | ✅ Training curves saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/training_curves.png


INFO:SENTIRA-Audio:✅ Training curves saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/training_curves.png


## **PHASE 6 · FULL INFERENCE ON TEST SET**

In [18]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 6 · FULL INFERENCE ON TEST SET                        ║
# ╚══════════════════════════════════════════════════════════════╝

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, confusion_matrix
)

# ── Run inference ──
model.eval()
y_true, y_pred   = [], []
y_probs_all      = []
attention_weights = []

with torch.no_grad():
    for trad, w2v, sem, labels in test_loader:
        trad   = trad.to(DEVICE)
        w2v    = w2v.to(DEVICE)
        sem    = sem.to(DEVICE)

        logits, attn = model(trad, w2v, sem, return_attn=True)
        probs  = torch.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs_all.extend(probs.cpu().numpy())
        attention_weights.extend(attn.cpu().numpy())

y_true           = np.array(y_true)
y_pred           = np.array(y_pred)
y_probs_all      = np.array(y_probs_all)
attention_weights = np.array(attention_weights)

# ── Compute Metrics ──
accuracy    = accuracy_score(y_true, y_pred) * 100
macro_f1    = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
kappa       = cohen_kappa_score(y_true, y_pred)

print("\n" + "="*60)
print("  SENTIRA UC1 — AUDIO EMOTION RECOGNITION")
print("  FINAL TEST SET EVALUATION (Unseen Subjects)")
print("="*60)
print(f"\n  Overall Accuracy  : {accuracy:.2f}%")
print(f"  Macro-F1          : {macro_f1:.4f}  ({macro_f1*100:.2f}%)")
print(f"  Weighted-F1       : {weighted_f1:.4f}  ({weighted_f1*100:.2f}%)")
print(f"  Cohen's κ         : {kappa:.4f}")
print(f"\n  Test Samples      : {len(y_true)}")
print(f"  Test Subjects     : {len(split_info['test'])}")
print("="*60)

# ── Per-class report ──
target_names = [Config.EMOTION_NAMES[Config.REVERSE_MAP[i]] for i in range(5)]
print("\n📊 Per-Class Classification Report:")
print(classification_report(y_true, y_pred,
                             target_names=target_names,
                             digits=4))


  SENTIRA UC1 — AUDIO EMOTION RECOGNITION
  FINAL TEST SET EVALUATION (Unseen Subjects)

  Overall Accuracy  : 97.50%
  Macro-F1          : 0.9750  (97.50%)
  Weighted-F1       : 0.9750  (97.50%)
  Cohen's κ         : 0.9688

  Test Samples      : 600
  Test Subjects     : 6

📊 Per-Class Classification Report:
              precision    recall  f1-score   support

   Happiness     0.9917    1.0000    0.9959       120
     Sadness     0.9360    0.9750    0.9551       120
       Angry     0.9825    0.9333    0.9573       120
    Calmness     0.9677    1.0000    0.9836       120
     Neutral     1.0000    0.9667    0.9831       120

    accuracy                         0.9750       600
   macro avg     0.9756    0.9750    0.9750       600
weighted avg     0.9756    0.9750    0.9750       600



## **PHASE 6 · CONFUSION MATRIX (Publication Quality 300 DPI)**

In [19]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 6 · CONFUSION MATRIX (Publication Quality 300 DPI)    ║
# ╚══════════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np

emotion_labels = [Config.EMOTION_NAMES[Config.REVERSE_MAP[i]] for i in range(5)]

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    f"SENTIRA UC1 — Confusion Matrix | Accuracy: {accuracy:.2f}% | κ={kappa:.4f}",
    fontsize=14, fontweight="bold"
)

# ── Raw Counts ──
ax = axes[0]
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=emotion_labels, yticklabels=emotion_labels,
    ax=ax, linewidths=0.5, linecolor="grey",
    cbar_kws={"label": "Sample Count"}
)
ax.set_title("Raw Counts", fontsize=12, fontweight="bold")
ax.set_xlabel("Predicted Emotion", fontsize=10)
ax.set_ylabel("True Emotion",      fontsize=10)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)

# ── Normalized (%) ──
ax = axes[1]
sns.heatmap(
    cm_norm * 100, annot=True, fmt=".1f", cmap="Blues",
    xticklabels=emotion_labels, yticklabels=emotion_labels,
    ax=ax, linewidths=0.5, linecolor="grey",
    vmin=0, vmax=100,
    cbar_kws={"label": "Recall (%)"}
)
ax.set_title("Normalized (Row = Recall %)", fontsize=12, fontweight="bold")
ax.set_xlabel("Predicted Emotion", fontsize=10)
ax.set_ylabel("True Emotion",      fontsize=10)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
cm_path = Config.DRIVE_FIGURES_DIR / "confusion_matrix_UC1.png"
plt.savefig(cm_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
plt.show()
LOG.info(f"✅ Confusion matrix saved: {cm_path}")

10:22:39 | INFO | ✅ Confusion matrix saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/confusion_matrix_UC1.png


INFO:SENTIRA-Audio:✅ Confusion matrix saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/confusion_matrix_UC1.png


## **PHASE 6 · PER-EMOTION F1 BAR CHART (300 DPI)**

In [20]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 6 · PER-EMOTION F1 BAR CHART (300 DPI)                ║
# ╚══════════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred,
                                target_names=emotion_labels,
                                output_dict=True)

metrics = {
    "Precision": [report[e]["precision"] * 100 for e in emotion_labels],
    "Recall":    [report[e]["recall"]    * 100 for e in emotion_labels],
    "F1-Score":  [report[e]["f1-score"]  * 100 for e in emotion_labels],
}

x      = np.arange(len(emotion_labels))
width  = 0.25
colors = ["#2196F3", "#4CAF50", "#FF9800"]

fig, ax = plt.subplots(figsize=(13, 6))

for i, (metric, vals) in enumerate(metrics.items()):
    bars = ax.bar(x + i * width, vals, width, label=metric,
                  color=colors[i], alpha=0.85, edgecolor="white")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f"{h:.1f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xlabel("Emotion Class", fontsize=11)
ax.set_ylabel("Score (%)",     fontsize=11)
ax.set_title(
    f"SENTIRA UC1 — Per-Emotion Classification Metrics\n"
    f"Macro-F1: {macro_f1*100:.2f}% | Weighted-F1: {weighted_f1*100:.2f}% | Accuracy: {accuracy:.2f}%",
    fontsize=12, fontweight="bold"
)
ax.set_xticks(x + width)
ax.set_xticklabels(emotion_labels, fontsize=10)
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.axhline(y=99, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Target (99%)")

plt.tight_layout()
f1_path = Config.DRIVE_FIGURES_DIR / "per_emotion_metrics_UC1.png"
plt.savefig(f1_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
plt.show()
LOG.info(f"✅ Per-emotion chart saved: {f1_path}")

10:22:42 | INFO | ✅ Per-emotion chart saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/per_emotion_metrics_UC1.png


INFO:SENTIRA-Audio:✅ Per-emotion chart saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/per_emotion_metrics_UC1.png


## **PHASE 6 · ATTENTION WEIGHTS VISUALIZATION**

In [21]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 6 · ATTENTION WEIGHTS VISUALIZATION                   ║
# ╚══════════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import numpy as np

stream_labels = ["Traditional\nAcoustic", "Wav2Vec2-L\n(Speech)", "Semantic\n(RoBERTa)"]

# ── Mean attention per emotion class ──
mean_attn_per_emotion = {}
for emo_idx in range(5):
    mask = y_true == emo_idx
    if mask.sum() > 0:
        mean_attn_per_emotion[emo_idx] = attention_weights[mask].mean(axis=0)

# ── Heatmap: Emotion × Stream ──
attn_matrix = np.array([
    mean_attn_per_emotion.get(i, np.zeros(3)) for i in range(5)
])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("SENTIRA UC1 — Branch Attention Weight Analysis",
             fontsize=13, fontweight="bold")

# Heatmap
ax = axes[0]
im = ax.imshow(attn_matrix, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(3));   ax.set_xticklabels(stream_labels, fontsize=9)
ax.set_yticks(range(5));   ax.set_yticklabels(emotion_labels, fontsize=9)
ax.set_title("Mean Attention Weight per Emotion × Stream")
plt.colorbar(im, ax=ax, label="Attention Weight")
for i in range(5):
    for j in range(3):
        ax.text(j, i, f"{attn_matrix[i,j]:.3f}",
                ha="center", va="center", fontsize=9,
                color="black" if attn_matrix[i,j] < 0.5 else "white")

# Radar / Polar
ax = axes[1]
overall_mean = attention_weights.mean(axis=0)
bars = ax.bar(range(3), overall_mean * 100,
              color=["#2196F3", "#4CAF50", "#FF9800"],
              alpha=0.8, edgecolor="white")
ax.set_xticks(range(3));  ax.set_xticklabels(stream_labels, fontsize=9)
ax.set_ylabel("Mean Attention Weight (%)")
ax.set_title(f"Overall Mean Attention (n={len(y_true)} samples)")
ax.set_ylim(0, 70)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1,
            f"{h:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
attn_path = Config.DRIVE_FIGURES_DIR / "attention_weights_UC1.png"
plt.savefig(attn_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
plt.show()
LOG.info(f"✅ Attention weights figure saved: {attn_path}")

10:22:49 | INFO | ✅ Attention weights figure saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/attention_weights_UC1.png


INFO:SENTIRA-Audio:✅ Attention weights figure saved: /content/drive/MyDrive/THESIS/Audio Results/Figures/audio/attention_weights_UC1.png


## **PHASE 6 · STATISTICAL TESTS (IEEE MANDATORY)**

In [22]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 6 · STATISTICAL TESTS (IEEE MANDATORY)                ║
# ╚══════════════════════════════════════════════════════════════╝
#
#  McNemar's test framework for cross-UC comparison (UC1 vs UC2–UC7).
#  Saves predictions so you can compare after training other UCs.
#
#  IEEE reviewers require p < 0.05 for statistical significance.

import numpy as np
from pathlib import Path

PREDS_DRIVE_PATH = Config.DRIVE_RESULTS_DIR / "uc1_predictions.npz"

# Save predictions to Drive for cross-UC McNemar tests
np.savez(
    PREDS_DRIVE_PATH,
    y_true       = y_true,
    y_pred       = y_pred,
    y_probs      = y_probs_all,
    test_subjects= np.array(split_info["test"], dtype=str),
)
LOG.info(f"✅ Predictions saved for cross-UC comparison: {PREDS_DRIVE_PATH}")


def mcnemar_test(preds_a: np.ndarray, preds_b: np.ndarray,
                 y_true: np.ndarray, label: str = "McNemar"):
    """
    McNemar's test comparing two classifiers on the SAME test set.
    Use this after training UC2–UC7 to compare against UC1.

    H0: Both classifiers have the same error rate.
    p < 0.05 → statistically significant improvement.
    """
    from statsmodels.stats.contingency_tables import mcnemar
    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)

    # Contingency table: [[both_correct, only_a_correct], [only_b_correct, both_wrong]]
    n00 = ((~correct_a) & (~correct_b)).sum()
    n01 = ((~correct_a) & correct_b).sum()
    n10 = (correct_a    & (~correct_b)).sum()
    n11 = (correct_a    & correct_b).sum()

    table  = np.array([[n11, n10], [n01, n00]])
    result = mcnemar(table, exact=False, correction=True)

    print(f"\n  {label}:")
    print(f"    Both correct:     {n11}")
    print(f"    Only A correct:   {n10}")
    print(f"    Only B correct:   {n01}")
    print(f"    Both wrong:       {n00}")
    print(f"    χ² statistic: {result.statistic:.4f}")
    print(f"    p-value:      {result.pvalue:.4e}  {'✅ SIGNIFICANT' if result.pvalue < 0.05 else '❌ Not significant'}")
    return result


def friedman_test(*pred_arrays, y_true):
    """
    Friedman test for comparing all 7 UC classifiers simultaneously.
    Use after all UCs are evaluated.
    """
    from scipy.stats import friedmanchisquare
    binary_results = [(preds == y_true).astype(int) for preds in pred_arrays]
    stat, pvalue = friedmanchisquare(*binary_results)
    print(f"\n  Friedman Test (all UCs):")
    print(f"    χ² statistic: {stat:.4f}")
    print(f"    p-value:      {pvalue:.4e}  {'✅ SIGNIFICANT' if pvalue < 0.05 else '❌ Not significant'}")
    return stat, pvalue


print("=" * 60)
print("  STATISTICAL TEST FRAMEWORK READY")
print("=" * 60)
print("\n  UC1 predictions saved to Drive.")
print("\n  After training UC2–UC7, use:")
print("    from pathlib import Path")
print("    import numpy as np")
print("    uc1 = np.load(PREDS_DRIVE_PATH)")
print("    uc7 = np.load(DRIVE_RESULTS_DIR / 'uc7_predictions.npz')")
print("    mcnemar_test(uc1['y_pred'], uc7['y_pred'], uc1['y_true'], 'UC1 vs UC7')")

10:22:49 | INFO | ✅ Predictions saved for cross-UC comparison: /content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_predictions.npz


INFO:SENTIRA-Audio:✅ Predictions saved for cross-UC comparison: /content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_predictions.npz


  STATISTICAL TEST FRAMEWORK READY

  UC1 predictions saved to Drive.

  After training UC2–UC7, use:
    from pathlib import Path
    import numpy as np
    uc1 = np.load(PREDS_DRIVE_PATH)
    uc7 = np.load(DRIVE_RESULTS_DIR / 'uc7_predictions.npz')
    mcnemar_test(uc1['y_pred'], uc7['y_pred'], uc1['y_true'], 'UC1 vs UC7')


## **PHASE 6 · SAVE COMPLETE RESULTS (IEEE PAPER TABLE)**

In [23]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 6 · SAVE COMPLETE RESULTS (IEEE PAPER TABLE)          ║
# ╚══════════════════════════════════════════════════════════════╝

import json
from sklearn.metrics import classification_report
from datetime import datetime

report_dict = classification_report(y_true, y_pred,
                                     target_names=emotion_labels,
                                     output_dict=True)

results = {
    "use_case":      "UC1",
    "description":   "Audio Only — Wav2Vec2-Large + Whisper + RoBERTa",
    "dataset":       "EAV",
    "n_test_samples": int(len(y_true)),
    "n_test_subjects": len(split_info["test"]),
    "test_subjects": split_info["test"],
    "seed":          Config.SEED,
    "evaluated_at":  datetime.now().isoformat(),

    # ── Primary Metrics (IEEE Paper Table) ──
    "accuracy":      round(accuracy, 4),
    "macro_f1":      round(macro_f1, 4),
    "weighted_f1":   round(weighted_f1, 4),
    "cohen_kappa":   round(kappa, 4),

    # ── Per-class Metrics ──
    "per_class": {
        emo: {
            "precision": round(report_dict[emo]["precision"], 4),
            "recall":    round(report_dict[emo]["recall"],    4),
            "f1_score":  round(report_dict[emo]["f1-score"],  4),
            "support":   int(report_dict[emo]["support"]),
        }
        for emo in emotion_labels
    },

    # ── Confusion Matrix ──
    "confusion_matrix": cm.tolist(),
    "confusion_matrix_normalized": (cm.astype(float) / cm.sum(axis=1, keepdims=True)).round(4).tolist(),

    # ── Model Config ──
    "model": {
        "architecture": "AttentionFusionModel",
        "trad_dim":     TRAD_DIM,
        "w2v_dim":      W2V_DIM,
        "sem_dim":      SEM_DIM,
        "hidden_dim":   Config.HIDDEN_DIM,
        "n_params":     sum(p.numel() for p in model.parameters()),
        "dropout":      Config.DROPOUT,
        "wav2vec_model": Config.WAV2VEC_MODEL,
        "whisper_model": Config.WHISPER_MODEL,
        "roberta_model": Config.ROBERTA_MODEL,
    },

    # ── Attention (for interpretation) ──
    "mean_attention_weights": {
        "traditional": float(attention_weights[:, 0].mean()),
        "wav2vec":     float(attention_weights[:, 1].mean()),
        "semantic":    float(attention_weights[:, 2].mean()),
    },
}

RESULTS_PATH = Config.DRIVE_RESULTS_DIR / "uc1_results.json"
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

LOG.info(f"✅ Complete results saved: {RESULTS_PATH}")

# ── Print IEEE Paper Table Row ──
print("\n" + "="*70)
print("  IEEE PAPER — RESULTS TABLE (UC1 Row)")
print("="*70)
print(f"  | UC1 | ✓ | ✗ | ✗ | {accuracy:.2f}% | {macro_f1:.4f} | {kappa:.4f} |")
print("="*70)
print("\n  Per-emotion F1 for thesis Table 4:")
for emo in emotion_labels:
    f1 = report_dict[emo]["f1-score"]
    print(f"    {emo:<12}: F1={f1:.4f}  Prec={report_dict[emo]['precision']:.4f}  Rec={report_dict[emo]['recall']:.4f}")

10:22:50 | INFO | ✅ Complete results saved: /content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_results.json


INFO:SENTIRA-Audio:✅ Complete results saved: /content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_results.json



  IEEE PAPER — RESULTS TABLE (UC1 Row)
  | UC1 | ✓ | ✗ | ✗ | 97.50% | 0.9750 | 0.9688 |

  Per-emotion F1 for thesis Table 4:
    Happiness   : F1=0.9959  Prec=0.9917  Rec=1.0000
    Sadness     : F1=0.9551  Prec=0.9360  Rec=0.9750
    Angry       : F1=0.9573  Prec=0.9825  Rec=0.9333
    Calmness    : F1=0.9836  Prec=0.9677  Rec=1.0000
    Neutral     : F1=0.9831  Prec=1.0000  Rec=0.9667


## **PHASE 7 · FUSION-READY INFERENCE WRAPPER**

In [24]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 7 · FUSION-READY INFERENCE WRAPPER                    ║
# ╚══════════════════════════════════════════════════════════════╝
#
#  This class is the Audio branch of the SENTIRA FusionEngine.
#  Usage in FusionEngine:
#
#   audio_wrapper = AudioFusionWrapper.load(drive_path)
#   probs = audio_wrapper.predict_proba(audio_path)   # [5] tensor
#   conf  = audio_wrapper.get_confidence(audio_path)  # scalar
#
#  The FusionEngine then combines:
#   P_fused = w_audio * P_audio + w_video * P_video + w_eeg * P_eeg
#   where w_i = confidence_i (if above threshold 0.3, else 0)

import pickle, torch
from pathlib import Path

class AudioFusionWrapper:
    """
    SENTIRA Audio Branch Inference Wrapper.
    Encapsulates: feature extraction models + scalers + attention model.
    Thread-safe for multi-modal inference.
    """

    LABEL_MAP_INT = {0:"Happiness", 1:"Sadness", 2:"Angry", 3:"Calmness", 4:"Neutral"}

    def __init__(
        self,
        model:    AttentionFusionModel,
        scalers:  dict,
        device:   torch.device,
        extractor: MultiStreamExtractor,
        config:   type = Config,
    ):
        self.model    = model.eval()
        self.scalers  = scalers
        self.device   = device
        self.extractor = extractor
        self.config   = config

    @torch.no_grad()
    def _extract(self, audio_path: str):
        """Full pipeline: audio file → normalized feature tensors."""
        audio = load_audio(audio_path)
        if audio is None:
            raise ValueError(f"Cannot load audio: {audio_path}")

        trad = extract_traditional(audio)
        w2v  = self.extractor.extract_wav2vec(audio)
        sem, _ = self.extractor.extract_semantic(audio)

        # Normalize with training scalers
        trad = self.scalers["trad"].transform(trad.reshape(1, -1)).astype("float32")
        w2v  = self.scalers["w2v"].transform(w2v.reshape(1, -1)).astype("float32")
        sem  = self.scalers["sem"].transform(sem.reshape(1, -1)).astype("float32")

        return (
            torch.from_numpy(trad).to(self.device),
            torch.from_numpy(w2v).to(self.device),
            torch.from_numpy(sem).to(self.device),
        )

    @torch.no_grad()
    def predict_proba(self, audio_path: str) -> torch.Tensor:
        """
        Returns softmax probability vector [5].
        Used by FusionEngine for weighted fusion.
        """
        trad, w2v, sem = self._extract(audio_path)
        probs = self.model.get_softmax_probs(trad, w2v, sem)
        return probs.squeeze(0)   # [5]

    @torch.no_grad()
    def get_confidence(self, audio_path: str) -> float:
        """
        Returns max(softmax) as confidence scalar.
        Used as fusion weight w_audio in FusionEngine.
        """
        probs = self.predict_proba(audio_path)
        return float(probs.max().item())

    @torch.no_grad()
    def predict(self, audio_path: str) -> dict:
        """
        Full prediction: class + probabilities + confidence.
        Returns dict for logging and debugging.
        """
        probs      = self.predict_proba(audio_path)
        pred_idx   = int(probs.argmax().item())
        confidence = float(probs.max().item())
        return {
            "predicted_class": pred_idx,
            "predicted_emotion": self.LABEL_MAP_INT[pred_idx],
            "confidence": confidence,
            "probabilities": {
                self.LABEL_MAP_INT[i]: float(probs[i].item()) for i in range(5)
            }
        }

    def save(self, path: Path):
        """Serialize wrapper (model + scalers) to pickle."""
        state = {
            "model_state_dict": self.model.state_dict(),
            "model_config": {
                "trad_dim":   self.model.trad_dim,
                "w2v_dim":    self.model.w2v_dim,
                "sem_dim":    self.model.sem_dim,
                "hidden_dim": self.model.hidden_dim,
            },
            "scalers": self.scalers,
        }
        with open(path, "wb") as f:
            pickle.dump(state, f)
        LOG.info(f"✅ AudioFusionWrapper saved: {path}")

    @classmethod
    def load(cls, path: Path, device: torch.device,
             extractor: "MultiStreamExtractor") -> "AudioFusionWrapper":
        """Reload wrapper from pickle — for use in FusionEngine."""
        with open(path, "rb") as f:
            state = pickle.load(f)
        m = AttentionFusionModel(**state["model_config"]).to(device)
        m.load_state_dict(state["model_state_dict"])
        m.eval()
        return cls(
            model=m, scalers=state["scalers"],
            device=device, extractor=extractor
        )


# ── Instantiate wrapper ──
# Note: extractor already loaded (or reload for CPU inference)
try:
    _ext = extractor   # from extraction phase
except NameError:
    _ext = MultiStreamExtractor()
    _ext.load_models()

wrapper = AudioFusionWrapper(
    model=model, scalers=scalers,
    device=DEVICE, extractor=_ext
)

# ── Save to Drive ──
WRAPPER_DRIVE_PATH = Config.DRIVE_MODELS_DIR / "audio_fusion_wrapper.pkl"
wrapper.save(WRAPPER_DRIVE_PATH)
LOG.info(f"✅ FusionWrapper saved: {WRAPPER_DRIVE_PATH}")

10:22:50 | INFO | 📥 Loading Wav2Vec2-Large ...


INFO:SENTIRA-Audio:📥 Loading Wav2Vec2-Large ...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

10:23:50 | INFO | ✅ AudioFusionWrapper saved: /content/drive/MyDrive/THESIS/Audio Results/Models/audio/audio_fusion_wrapper.pkl


INFO:SENTIRA-Audio:✅ AudioFusionWrapper saved: /content/drive/MyDrive/THESIS/Audio Results/Models/audio/audio_fusion_wrapper.pkl


10:23:50 | INFO | ✅ FusionWrapper saved: /content/drive/MyDrive/THESIS/Audio Results/Models/audio/audio_fusion_wrapper.pkl


INFO:SENTIRA-Audio:✅ FusionWrapper saved: /content/drive/MyDrive/THESIS/Audio Results/Models/audio/audio_fusion_wrapper.pkl


## **PHASE 7 · VALIDATE FUSION WRAPPER**

In [25]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 7 · VALIDATE FUSION WRAPPER                           ║
# ╚══════════════════════════════════════════════════════════════╝

import random

# Pick a random test file
test_subject = random.choice(split_info["test"])
test_sub_dir = Config.LOCAL_AUDIO_DIR / test_subject

audio_sub = test_sub_dir / "Audio"
if not audio_sub.exists():
    audio_sub = test_sub_dir
test_files = list(audio_sub.glob("*.wav")) + list(audio_sub.glob("*.WAV"))

if test_files:
    test_file = str(random.choice(test_files))
    print(f"🎵 Test file: {test_file}")

    result = wrapper.predict(test_file)
    print(f"\n  Predicted Emotion : {result['predicted_emotion']}")
    print(f"  Confidence        : {result['confidence']:.4f}")
    print(f"\n  Full Probability Vector:")
    for emo, prob in result["probabilities"].items():
        bar = "█" * int(prob * 40)
        print(f"    {emo:<12}: {prob:.4f}  {bar}")

    print(f"\n  ✅ Confidence gate check (threshold=0.3): "
          f"{'PASS — use in fusion' if result['confidence'] >= 0.3 else 'FAIL — exclude from fusion'}")
else:
    LOG.warning(f"No .wav files found in {test_sub_dir}. Skipping wrapper test.")

10:23:50 | WARNING | No .wav files found in /content/audio_data/Audio/subject41. Skipping wrapper test.


## **PHASE 7 · FINAL DRIVE SYNC — ALL ARTIFACTS**

In [26]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PHASE 7 · FINAL DRIVE SYNC — ALL ARTIFACTS                  ║
# ╚══════════════════════════════════════════════════════════════╝

import shutil
from pathlib import Path
import json
from datetime import datetime

# ── Summary manifest ──
manifest = {
    "project":  "SENTIRA",
    "use_case": "UC1 — Audio Only",
    "created":  datetime.now().isoformat(),
    "files": {
        "hdf5_features":      str(Config.DRIVE_FEATURES_DIR / "audio_features.h5"),
        "best_model":         str(Config.DRIVE_MODELS_DIR / "best_audio_model.pt"),
        "scalers":            str(Config.DRIVE_MODELS_DIR / "scalers.pkl"),
        "subject_split":      str(Config.DRIVE_MODELS_DIR / "subject_split.json"),
        "training_history":   str(Config.DRIVE_MODELS_DIR / "training_history.json"),
        "fusion_wrapper":     str(Config.DRIVE_MODELS_DIR / "audio_fusion_wrapper.pkl"),
        "results_json":       str(Config.DRIVE_RESULTS_DIR / "uc1_results.json"),
        "predictions_npz":    str(Config.DRIVE_RESULTS_DIR / "uc1_predictions.npz"),
        "figure_cm":          str(Config.DRIVE_FIGURES_DIR / "confusion_matrix_UC1.png"),
        "figure_f1":          str(Config.DRIVE_FIGURES_DIR / "per_emotion_metrics_UC1.png"),
        "figure_attention":   str(Config.DRIVE_FIGURES_DIR / "attention_weights_UC1.png"),
        "figure_training":    str(Config.DRIVE_FIGURES_DIR / "training_curves.png"),
    },
    "metrics": {
        "accuracy":   round(accuracy, 4),
        "macro_f1":   round(macro_f1, 4),
        "weighted_f1":round(weighted_f1, 4),
        "cohen_kappa":round(kappa, 4),
    }
}

MANIFEST_PATH = Config.DRIVE_RESULTS_DIR / "uc1_manifest.json"
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

LOG.info("\n" + "=" * 65)
LOG.info("  ✅ SENTIRA UC1 — COMPLETE. All artifacts saved to Drive.")
LOG.info("=" * 65)
for k, v in manifest["files"].items():
    exists = Path(v).exists()
    mark   = "✅" if exists else "❌"
    LOG.info(f"  {mark}  {k:<25}: {v}")
LOG.info("\n  Final Metrics:")
for k, v in manifest["metrics"].items():
    LOG.info(f"     {k:<15}: {v}")
LOG.info("\n  Next step: Train UC2 (Video) and UC3 (EEG), then build FusionEngine.")
LOG.info("=" * 65)

10:23:50 | INFO | 


INFO:SENTIRA-Audio:


10:23:50 | INFO |   ✅ SENTIRA UC1 — COMPLETE. All artifacts saved to Drive.


INFO:SENTIRA-Audio:  ✅ SENTIRA UC1 — COMPLETE. All artifacts saved to Drive.


10:23:50 | INFO | =================================================================


...

INFO:SENTIRA-Audio:     cohen_kappa    : 0.9688


10:23:50 | INFO | 
  Next step: Train UC2 (Video) and UC3 (EEG), then build FusionEngine.


INFO:SENTIRA-Audio:
  Next step: Train UC2 (Video) and UC3 (EEG), then build FusionEngine.


10:23:50 | INFO | =================================================================


INFO:SENTIRA-Audio:=================================================================




---
## 📋 Summary & Next Steps

### ✅ UC1 Completed — What Was Produced

| Artifact | Description |
|---|---|
| `audio_features.h5` | 4,200 samples × 3 feature streams, compressed HDF5 |
| `best_audio_model.pt` | Best model weights (PyTorch state dict) |
| `scalers.pkl` | StandardScalers fitted on training data only |
| `subject_split.json` | Reproducibility lock: same 30/6/6 split always |
| `audio_fusion_wrapper.pkl` | Plug-in module for SENTIRA FusionEngine |
| `uc1_predictions.npz` | Test set predictions for McNemar tests |
| `uc1_results.json` | Complete metrics for IEEE paper Table |
| Figures (4×) | Confusion matrix, F1 chart, attention, training curves |

---
